# Re-scoring State's published predictions on Parse (cell-type splits)

State's public repository (`arcinstitute/ST-SE-Parse`) contains its own predicted and real test cells for each split. In each split, one cell type (split 0: **B_Intermediate_Memory**, split 1: **B_Naive**) is held out for 62 test cytokines (28 observed), for all 12 donors; the reference is PBS.

We compare, on the same split and with one common truth:
- **State** (published predictions)
- **additive**: mean over training contexts for the cytokine + the query context's panel offset
- **context mean**: mean of the query context's observed panel deltas (no cytokine information)
- **inert shift**: mean delta of inert cytokines over training contexts (no cytokine or context information)

Scored with `pearson_delta` as published (against PBS) and after **inert re-referencing** (in each context, the mean over the 8 inert test cytokines is subtracted from truth and from each method's own predictions).

**Truth:** per-cell normalization to 10,000 counts + log1p on the 2,000 genes used by State, averaged per donor x cell type x cytokine, computed once from the raw Parse file (the same space as State's `adata_real`, checked below).

In [2]:
import os, pickle, json
import numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad
from huggingface_hub import hf_hub_download

SPLITS = {"split_0": "B_Intermediate_Memory", "split_1": "B_Naive", "split_2": "Plasmablast",
          "split_4": "CD14_Mono"}   # split_3 (CD4_Memory) has no published predictions in the repository
ST_DIR = "ST-SE-Parse/fewshot/split_0"                                  # used for the gene list only
PARSE = "data/parse10m/20260203_Parse_10M_PBMC_cytokines.h5ad"
CACHE = "data/parse10m/cache/state_space_means.npz"
RESULTS_DIR = "results/state_rescoring_strict"
os.makedirs(RESULTS_DIR, exist_ok=True)
CONTROL = "PBS"
INERT = ["TPO", "Megalin", "PSPN", "EPO", "EGF"]
#INERT = ["Noggin", "Decorin", "PSPN", "GDNF", "Megalin", "PRL", "EPO", "TPO", "FGF-beta", "EGF", "HGF", "VEGF", "IGF-1"]
TEST = ['C5a', 'TWEAK', 'LIF', 'IL-17C', 'FGF-beta', 'BAFF', 'FLT3L', 'IFN-lambda2', '4-1BBL', 'IL-17E', 'IL-2',
        'IL-19', 'PRL', 'IL-16', 'IL-7', 'GM-CSF', 'IL-17A', 'IL-13', 'IL-36-alpha', 'PSPN', 'EPO', 'IL-9', 'IL-31',
        'IL-11', 'LT-alpha1-beta2', 'TL1A', 'IL-32-beta', 'IFN-omega', 'IL-8', 'Noggin', 'IFN-epsilon', 'IL-17F',
        'Leptin', 'IL-33', 'IL-35', 'OSM', 'IL-34', 'HGF', 'IL-1Ra', 'LIGHT', 'CD27L', 'SCF', 'IL-1-alpha',
        'TGF-beta1', 'CD30L', 'IL-24', 'IFN-gamma', 'IFN-alpha1', 'IFN-lambda1', 'GITRL', 'VEGF', 'ADSF', 'IL-15',
        'EGF', 'IL-3', 'IL-17B', 'IL-6', 'IL-4', 'IL-10', 'CT-1', 'IFN-lambda3', 'TRAIL']
VAL = ['G-CSF', 'IFN-beta', 'M-CSF', 'IGF-1']

## 1. The 2,000 genes used by State

In [3]:
p = os.path.join(ST_DIR, "var_dims.pkl")
if not os.path.exists(p):
    hf_hub_download("arcinstitute/ST-SE-Parse", "fewshot/split_0/var_dims.pkl", local_dir="ST-SE-Parse")
var_dims = pickle.load(open(p, "rb"))
print("var_dims keys:", list(var_dims.keys()) if isinstance(var_dims, dict) else type(var_dims))
GENES = None
if isinstance(var_dims, dict):
    for k in ["gene_names", "hvg_names", "genes", "var_names"]:
        if k in var_dims and len(var_dims[k]) == 2000:
            GENES = [str(g) for g in var_dims[k]]; print("gene names from var_dims:", k); break
if GENES is None and os.path.exists("Rhaister/splits/parse/static_2k_genes.json"):
    g = json.load(open("Rhaister/splits/parse/static_2k_genes.json"))
    g = g if isinstance(g, list) else next(v for v in g.values() if isinstance(v, list))
    if len(g) == 2000: GENES = [str(x) for x in g]; print("gene names from Rhaister static_2k_genes.json")
assert GENES is not None, "could not find the 2,000 gene names; send the var_dims keys above"
print(GENES[:8])

var_dims keys: ['input_dim', 'gene_dim', 'hvg_dim', 'output_dim', 'pert_dim', 'gene_names', 'batch_dim', 'pert_names']
gene names from Rhaister static_2k_genes.json
['CFH', 'GCLC', 'LAP3', 'HS3ST1', 'AOC1', 'KLHL13', 'TFPI', 'CD38']


## 2. Truth in State's space from the raw Parse file (one pass, cached)

In [4]:
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    MEANS, KEYS, NC = z["means"], [tuple(k) for k in z["keys"]], z["n"]
    print("loaded", MEANS.shape)
else:
    a = ad.read_h5ad(PARSE, backed="r")
    gidx = pd.Index(a.var_names).get_indexer(GENES)
    assert (gidx >= 0).all(), f"{(gidx < 0).sum()} State genes not found in the Parse file"
    obs = a.obs[["donor", "cell_type", "cytokine"]].astype(str)
    obs["cell_type"] = obs["cell_type"].str.replace(" ", "_").str.replace("/", "_")
    code_, uniq = pd.factorize(pd.Series(list(zip(obs.donor, obs.cell_type, obs.cytokine))))
    S = np.zeros((len(uniq), len(GENES))); N = np.bincount(code_, minlength=len(uniq))
    for s in range(0, a.n_obs, 200_000):
        e = min(s + 200_000, a.n_obs)
        X = a.X[s:e]; X = sp.csr_matrix(X) if not sp.issparse(X) else X.tocsr()
        tot = np.asarray(X.sum(1)).ravel(); tot[tot == 0] = 1
        Xg = (sp.diags(1e4 / tot) @ X)[:, gidx]; Xg.data = np.log1p(Xg.data)       # per-cell normalize + log1p
        G = sp.csr_matrix((np.ones(e - s), (code_[s:e], np.arange(e - s))), shape=(len(uniq), e - s))
        S += (G @ Xg).toarray()
        print(f"\r{e:,}/{a.n_obs:,}", end="")
    MEANS = S / np.maximum(N, 1)[:, None]; KEYS = list(uniq); NC = N
    np.savez(CACHE, means=MEANS, keys=np.array(KEYS, dtype=object), n=NC)
    print("\ncached", MEANS.shape)
M = {k: MEANS[i] for i, k in enumerate(KEYS) if NC[i] >= 10}
print("cell types:", sorted({k[1] for k in M})[:30])

loaded (19459, 2000)
cell types: ['B_Intermediate_Memory', 'B_Naive', 'CD14_Mono', 'CD16_Mono', 'CD4_Memory', 'CD4_Naive', 'CD8_Memory', 'CD8_Naive', 'HSPC', 'ILC', 'MAIT', 'NK', 'NKT', 'NK_CD56bright', 'Plasmablast', 'Treg', 'cDC', 'pDC']


## 2b. Download State's predictions for each split (split 0 about 7 GB, split 1 about 15 GB, split 2 about 0.1 GB, split 4 about 37 GB)
Files already downloaded are skipped.

In [5]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import snapshot_download
for split in SPLITS:
    snapshot_download("arcinstitute/ST-SE-Parse", local_dir="ST-SE-Parse", allow_patterns=[f"fewshot/{split}/eval_best.ckpt/adata_*.h5ad"])
    print(split, sorted(os.listdir(f"ST-SE-Parse/fewshot/{split}/eval_best.ckpt")))

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

split_0 ['B_Intermediate_Memory_agg_results.csv', 'B_Intermediate_Memory_pred_de.csv', 'B_Intermediate_Memory_real_de.csv', 'B_Intermediate_Memory_results.csv', 'adata_pred.h5ad', 'adata_real.h5ad']


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

split_1 ['B_Naive_agg_results.csv', 'B_Naive_pred_de.csv', 'B_Naive_real_de.csv', 'B_Naive_results.csv', 'adata_pred.h5ad', 'adata_real.h5ad']


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

split_2 ['Plasmablast_agg_results.csv', 'Plasmablast_pred_de.csv', 'Plasmablast_real_de.csv', 'Plasmablast_results.csv', 'adata_pred.h5ad', 'adata_real.h5ad']


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

split_4 ['CD14_Mono_agg_results.csv', 'CD14_Mono_pred_de.csv', 'CD14_Mono_real_de.csv', 'CD14_Mono_results.csv', 'adata_pred.h5ad', 'adata_real.h5ad']


## 3. Score each split (with a sanity check against State's own real cells)

In [6]:
def pearson(a, b):
    a = a - a.mean(); b = b - b.mean(); return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def group_means(path):
    a = ad.read_h5ad(path, backed="r")
    o = a.obs[["donor", "cytokine"]].astype(str)
    code_, uniq = pd.factorize(pd.Series(list(zip(o.donor, o.cytokine))))
    S = np.zeros((len(uniq), a.n_vars)); N = np.bincount(code_, minlength=len(uniq))
    for s in range(0, a.n_obs, 50_000):
        e = min(s + 50_000, a.n_obs)
        X = a.X[s:e]; X = X.toarray() if sp.issparse(X) else np.asarray(X)
        np.add.at(S, code_[s:e], X)
    return {k: S[i] / N[i] for i, k in enumerate(uniq)}

def delta(d, ct, c):
    a, b = M.get((d, ct, c)), M.get((d, ct, CONTROL))
    return None if a is None or b is None else a - b

def score_split(split, QUERY_CT):
    ev = f"ST-SE-Parse/fewshot/{split}/eval_best.ckpt"
    PRED_M = group_means(os.path.join(ev, "adata_pred.h5ad")); REAL_M = group_means(os.path.join(ev, "adata_real.h5ad"))
    donors = sorted({d for d, _ in REAL_M})
    chk = [pearson(delta(d, QUERY_CT, c), REAL_M[(d, c)] - REAL_M[(d, CONTROL)])
           for d in donors for c in TEST if delta(d, QUERY_CT, c) is not None and (d, c) in REAL_M and (d, CONTROL) in REAL_M]
    print(f"{split} ({QUERY_CT}): agreement with State's real cells, median r = {np.median(chk):.3f} over {len(chk)} contexts")
    test_c = [c for c in TEST if any((d, c) in REAL_M for d in donors)]
    panel = [c for c in sorted({k[2] for k in M}) if c not in TEST + VAL + [CONTROL]] + VAL
    train_ctx = sorted({(d, ct) for d, ct, _ in M if ct != QUERY_CT})
    TM = {}
    for c in set(test_c) | set(panel) | set(INERT):
        v = [delta(d, ct, c) for d, ct in train_ctx]; v = [x for x in v if x is not None]
        TM[c] = np.mean(v, 0) if v else None
    shift = np.mean([TM[k] for k in INERT if TM.get(k) is not None], 0)
    rows = []
    for d in donors:
        obs_panel = {p: delta(d, QUERY_CT, p) for p in panel}
        obs_panel = {p: v for p, v in obs_panel.items() if v is not None and TM.get(p) is not None}
        if len(obs_panel) < 5: continue
        ctx_mean = np.mean(list(obs_panel.values()), 0)
        offset = ctx_mean - np.mean([TM[p] for p in obs_panel], 0)
        preds, truth = {}, {}
        for c in test_c:
            t = delta(d, QUERY_CT, c)
            if t is None or (d, c) not in PRED_M or (d, CONTROL) not in REAL_M: continue
            truth[c] = t
            preds.setdefault("State", {})[c] = PRED_M[(d, c)] - REAL_M[(d, CONTROL)]
            preds.setdefault("additive", {})[c] = TM[c] + offset
            preds.setdefault("context mean", {})[c] = ctx_mean
            preds.setdefault("inert shift", {})[c] = shift
        inert_here = [c for c in truth if c in INERT]
        if len(inert_here) < 3: continue
        ref_t = np.mean([truth[c] for c in inert_here], 0)
        for name, P in preds.items():
            ref_p = np.mean([P[c] for c in inert_here], 0)
            for c in truth:
                if c in INERT: continue
                rows.append(dict(split=split, cell_type=QUERY_CT, model=name, donor=d, cytokine=c,
                                 pbs=pearson(P[c], truth[c]), inert=pearson(P[c] - ref_p, truth[c] - ref_t)))
    return pd.DataFrame(rows)

SC = pd.concat([score_split(s, ct) for s, ct in SPLITS.items()], ignore_index=True)
SC.to_csv(f"{RESULTS_DIR}/scores_per_context.csv", index=False)
SUMMARY = SC.groupby(["split", "cell_type", "model"])[["pbs", "inert"]].mean().round(3)
SUMMARY["rank_pbs"] = SUMMARY.groupby(level=0).pbs.rank(ascending=False).astype(int)
SUMMARY["rank_inert"] = SUMMARY.groupby(level=0).inert.rank(ascending=False).astype(int)
SUMMARY = SUMMARY.sort_values(["split", "pbs"], ascending=[True, False])
SUMMARY.to_csv(f"{RESULTS_DIR}/scores_summary.csv")
SUMMARY

split_0 (B_Intermediate_Memory): agreement with State's real cells, median r = 0.988 over 732 contexts
split_1 (B_Naive): agreement with State's real cells, median r = 0.990 over 732 contexts
split_2 (Plasmablast): agreement with State's real cells, median r = 0.978 over 103 contexts
split_4 (CD14_Mono): agreement with State's real cells, median r = 0.988 over 732 contexts


pbs  inert  rank_pbs  rank_inert
split   cell_type             model                                           
split_0 B_Intermediate_Memory additive      0.502  0.374         1           1
                              context mean  0.363 -0.000         2           3
                              State         0.274  0.321         3           2
                              inert shift   0.032  0.000         4           3
split_1 B_Naive               additive      0.480  0.376         1           2
                              context mean  0.345 -0.000         2           3
                              State         0.260  0.392         3           1
                              inert shift   0.034  0.000         4           3
split_4 CD14_Mono             additive      0.458  0.378         1           2
                              context mean  0.339  0.000         2           3
                              State         0.229  0.433         3           1
                              inert shift   0.044 -0.000         4           3

## 4. Paired test: State vs additive baseline
Per donor x cytokine differences, and per donor means (12 donors, the conservative unit), Wilcoxon signed-rank.

In [7]:
from scipy.stats import wilcoxon
rows = []
for split, g in SC.groupby("split"):
    for metric in ["pbs", "inert"]:
        w = g.pivot_table(index=["donor", "cytokine"], columns="model", values=metric)
        diff = (w["State"] - w["additive"]).dropna()
        by_donor = diff.groupby(level="donor").mean()
        rows.append(dict(split=split, cell_type=g.cell_type.iloc[0], metric=metric, state_minus_additive=diff.mean(),
                         p_pairs=wilcoxon(diff).pvalue, n_pairs=len(diff),
                         donors_state_better=f"{(by_donor > 0).sum()}/{len(by_donor)}",
                         p_donors=wilcoxon(by_donor).pvalue))
PAIRED = pd.DataFrame(rows).round(4)
PAIRED.to_csv(f"{RESULTS_DIR}/paired_state_vs_additive.csv", index=False)
PAIRED

,split,cell_type,metric,state_minus_additive,p_pairs,n_pairs,donors_state_better,p_donors
0,split_0,B_Intermediate_Memory,pbs,-0.2287,0.0000,696,0/12,0.0005
1,split_0,B_Intermediate_Memory,inert,-0.0528,0.0000,696,0/12,0.0005
2,split_1,B_Naive,pbs,-0.2193,0.0000,696,0/12,0.0005
3,split_1,B_Naive,inert,0.0157,0.4119,696,8/12,0.2036
4,split_4,CD14_Mono,pbs,-0.2282,0.0000,696,0/12,0.0005
5,split_4,CD14_Mono,inert,0.0551,0.0000,696,11/12,0.0010


In [9]:
# ===== 5. Sensitivity: subtract the OBSERVED inert mean from predictions as well (all four predictors, both inert sets) =====
# Main analysis ("inert"):  truth - observed inert mean;  prediction - the method's own predicted inert mean.
# Sensitivity ("inert_obs"): truth - observed inert mean;  prediction - observed inert mean.
# Needs sections 1-3 in memory (M, delta, pearson, group_means, SPLITS, TEST, VAL, CONTROL).
# State's prediction files are read once per split and reused for both inert sets.
from scipy.stats import wilcoxon
INERT_SETS = {"full": ["Noggin", "Decorin", "PSPN", "GDNF", "Megalin", "PRL", "EPO", "TPO", "FGF-beta", "EGF", "HGF", "VEGF", "IGF-1"],
              "strict": ["TPO", "Megalin", "PSPN", "EPO", "EGF"]}
OUT = "results/state_rescoring_sensitivity"; os.makedirs(OUT, exist_ok=True)
GM = {}
for split in SPLITS:
    ev = f"ST-SE-Parse/fewshot/{split}/eval_best.ckpt"
    GM[split] = (group_means(os.path.join(ev, "adata_pred.h5ad")), group_means(os.path.join(ev, "adata_real.h5ad")))
    print("read", split)

def score_split_sens(split, QUERY_CT, inert):
    PRED_M, REAL_M = GM[split]
    donors = sorted({d for d, _ in REAL_M})
    test_c = [c for c in TEST if any((d, c) in REAL_M for d in donors)]
    panel = [c for c in sorted({k[2] for k in M}) if c not in TEST + VAL + [CONTROL]] + VAL
    train_ctx = sorted({(d, ct) for d, ct, _ in M if ct != QUERY_CT})
    TM = {}
    for c in set(test_c) | set(panel) | set(inert):
        v = [delta(d, ct, c) for d, ct in train_ctx]; v = [x for x in v if x is not None]
        TM[c] = np.mean(v, 0) if v else None
    shift = np.mean([TM[k] for k in inert if TM.get(k) is not None], 0)
    rows = []
    for d in donors:
        obs_panel = {p: delta(d, QUERY_CT, p) for p in panel}
        obs_panel = {p: v for p, v in obs_panel.items() if v is not None and TM.get(p) is not None}
        if len(obs_panel) < 5: continue
        ctx_mean = np.mean(list(obs_panel.values()), 0)
        offset = ctx_mean - np.mean([TM[p] for p in obs_panel], 0)
        preds, truth = {}, {}
        for c in test_c:
            t = delta(d, QUERY_CT, c)
            if t is None or (d, c) not in PRED_M or (d, CONTROL) not in REAL_M: continue
            truth[c] = t
            preds.setdefault("State", {})[c] = PRED_M[(d, c)] - REAL_M[(d, CONTROL)]
            preds.setdefault("additive", {})[c] = TM[c] + offset
            preds.setdefault("context mean", {})[c] = ctx_mean
            preds.setdefault("inert shift", {})[c] = shift
        inert_here = [c for c in truth if c in inert]
        if len(inert_here) < 3: continue
        ref_t = np.mean([truth[c] for c in inert_here], 0)
        for name, P in preds.items():
            ref_p = np.mean([P[c] for c in inert_here], 0)
            for c in truth:
                if c in inert: continue
                rows.append(dict(split=split, cell_type=QUERY_CT, model=name, donor=d, cytokine=c,
                                 pbs=pearson(P[c], truth[c]),
                                 inert=pearson(P[c] - ref_p, truth[c] - ref_t),
                                 inert_obs=pearson(P[c] - ref_t, truth[c] - ref_t),
                                 inert_err=float(np.linalg.norm(ref_p - ref_t) / (np.linalg.norm(ref_t) + 1e-12))))
    return pd.DataFrame(rows)

summ, pair = [], []
for set_name, inert in INERT_SETS.items():
    S = pd.concat([score_split_sens(s, ct, inert) for s, ct in SPLITS.items()], ignore_index=True)
    S["inert_set"] = set_name
    S.to_csv(f"{OUT}/scores_{set_name}.csv", index=False)
    summ.append(S.groupby(["inert_set", "cell_type", "model"])[["pbs", "inert", "inert_obs", "inert_err"]].mean())
    for (split, ct), g in S.groupby(["split", "cell_type"]):
        for metric in ["inert", "inert_obs"]:
            w = g.pivot_table(index=["donor", "cytokine"], columns="model", values=metric)
            by_donor = (w["State"] - w["additive"]).dropna().groupby(level="donor").mean()
            if len(by_donor) < 6: continue
            pair.append(dict(inert_set=set_name, cell_type=ct, metric=metric, state_minus_additive=by_donor.mean(),
                             donors_state_better=f"{(by_donor > 0).sum()}/{len(by_donor)}", p_donors=wilcoxon(by_donor).pvalue))
SENS_SUMMARY = pd.concat(summ).round(3); SENS_SUMMARY.to_csv(f"{OUT}/summary.csv")
SENS_PAIRED = pd.DataFrame(pair).round(4); SENS_PAIRED.to_csv(f"{OUT}/paired.csv", index=False)
print(SENS_SUMMARY.to_string()); print(); print(SENS_PAIRED.to_string())
# inert_err: size of the error in each method's predicted inert mean, relative to the size of the observed inert mean

read split_0
read split_1
read split_2
read split_4
                                                pbs  inert  inert_obs  inert_err
inert_set cell_type             model                                           
full      B_Intermediate_Memory State         0.287  0.349      0.301      2.745
                                additive      0.509  0.397      0.477      0.923
                                context mean  0.372 -0.000      0.337      0.993
                                inert shift   0.035 -0.000      0.169      0.979
          B_Naive               State         0.275  0.429      0.296      3.313
                                additive      0.487  0.394      0.467      1.091
                                context mean  0.356  0.000      0.340      1.192
                                inert shift   0.043  0.000      0.185      0.972
          CD14_Mono             State         0.248  0.458      0.264      3.850
                                additive      0.466  0.38